# Measurement-Frequency Project Summary Dashboard

This notebook is an exploratory dashboard of the measurement-frequency experiments. It is not a paper-figure notebook and it does not rerun discovery, checkpoint selection, normalizer selection, prediction generation, or patient bootstrap logic.

The source notebooks are the authority for scientific definitions and calculations:

- `visualize_exp1_frequency.ipynb`
- `visualize_exp2a_horizon.ipynb`
- `visualize_exp2b_matched_count.ipynb`
- `visualize_exp3_frequency_shift.ipynb`
- `visualize_exp3_grud_frequency_shift.ipynb`
- `visualize_utils.py`

This summary consumes stable CSV exports written by those notebooks. If a source notebook has not yet been rerun after the export cells were added, its section is labeled `NOT AVAILABLE`. Partial seed sets are shown as partial, not suppressed.

Uncertainty conventions used throughout:

- Student-t CI across model seeds: run-to-run variability across trained seeds.
- Paired patient-level bootstrap CI: test-population uncertainty for paired prediction contrasts.

These intervals answer different questions and should not be silently mixed.


In [ ]:
from pathlib import Path
import ast
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

HERE = Path.cwd()
REPO_DIR = HERE.parent if HERE.name == "visualizers" else HERE
VIS_DIR = REPO_DIR / "visualizers"

PROJECT_DATA_ROOT = Path(os.environ.get(
    "MEAS_FREQ_DATA_ROOT",
    "/heinz-georgenas/users/mingzhul/Simultaneous-EHR/data/physionet.org/files/mimiciv/1.0/russo",
))
IHM_PLOTS = PROJECT_DATA_ROOT / "results/in-hospital-mortality/plots"
FH_PLOTS = PROJECT_DATA_ROOT / "results/fixed_horizon_icu_exit/plots"

EXPECTED_SEEDS = set(range(5))
METRICS = ["auprc", "auroc", "brier"]

EXPORTS = {
    "exp1_runs": IHM_PLOTS / "experiment1_runs.csv",
    "exp1_val_auprc_summary": IHM_PLOTS / "experiment1_validation_auprc_summary.csv",
    "exp1_val_auroc_summary": IHM_PLOTS / "experiment1_validation_auroc_summary.csv",
    "exp1_test_auprc_summary": IHM_PLOTS / "experiment1_test_auprc_summary.csv",
    "exp1_test_auroc_summary": IHM_PLOTS / "experiment1_test_auroc_summary.csv",
    "exp1_test_drops": IHM_PLOTS / "experiment1_test_degradation_seed_level.csv",
    "exp1_auprc_drop_summary": IHM_PLOTS / "experiment1_test_auprc_degradation_summary.csv",
    "exp1_auroc_drop_summary": IHM_PLOTS / "experiment1_test_auroc_degradation_summary.csv",

    "exp2a_val_runs": FH_PLOTS / "experiment2a_validation_runs.csv",
    "exp2a_val_auprc_summary": FH_PLOTS / "experiment2a_validation_auprc_summary.csv",
    "exp2a_val_auroc_summary": FH_PLOTS / "experiment2a_validation_auroc_summary.csv",
    "exp2a_val_auprc_drop_summary": FH_PLOTS / "experiment2a_validation_auprc_degradation_summary.csv",
    "exp2a_val_auroc_drop_summary": FH_PLOTS / "experiment2a_validation_auroc_degradation_summary.csv",
    "exp2a_test_runs": FH_PLOTS / "experiment2a_test_runs.csv",
    "exp2a_test_auprc_summary": FH_PLOTS / "experiment2a_test_auprc_summary.csv",
    "exp2a_test_auroc_summary": FH_PLOTS / "experiment2a_test_auroc_summary.csv",
    "exp2a_test_auprc_drop_summary": FH_PLOTS / "experiment2a_test_auprc_degradation_summary.csv",
    "exp2a_test_auroc_drop_summary": FH_PLOTS / "experiment2a_test_auroc_degradation_summary.csv",
    "exp2a_bootstrap": FH_PLOTS / "experiment2a_seed_level_patient_bootstrap.csv",
    "exp2a_bootstrap_summary": FH_PLOTS / "experiment2a_across_seed_patient_bootstrap_summary.csv",

    "exp2b_run_results": FH_PLOTS / "experiment2b_included_prediction_results.csv",
    "exp2b_condition_summary": FH_PLOTS / "experiment2b_condition_summary.csv",
    "exp2b_seed_level_contrasts": FH_PLOTS / "experiment2b_seed_level_patient_bootstrap_contrasts.csv",
    "exp2b_across_seed_summary": FH_PLOTS / "experiment2b_across_seed_contrast_summary.csv",

    "exp3_lstm_run_results": FH_PLOTS / "experiment3_lstm_condition_run_results.csv",
    "exp3_lstm_absolute_summary": FH_PLOTS / "experiment3_lstm_absolute_summary.csv",
    "exp3_lstm_information_summary": FH_PLOTS / "experiment3_lstm_information_effects_summary.csv",
    "exp3_lstm_structured_shift_summary": FH_PLOTS / "experiment3_lstm_structured_shift_summary.csv",
    "exp3_lstm_normalizer_control_summary": FH_PLOTS / "experiment3_lstm_normalizer_control_summary.csv",
    "exp3_lstm_random_shift_summary": FH_PLOTS / "experiment3_lstm_random_shift_summary.csv",
    "exp3_lstm_horizon_replication_summary": FH_PLOTS / "experiment3_lstm_horizon_replication_summary.csv",
    "exp3e_lstm_raw_absolute_summary": FH_PLOTS / "experiment3e_lstm_raw_absolute_summary.csv",
    "exp3e_lstm_raw_shift_summary": FH_PLOTS / "experiment3e_lstm_raw_shift_summary.csv",
    "exp3_lstm_grid_vs_raw_shift_summary": FH_PLOTS / "experiment3_lstm_grid_vs_raw_shift_summary.csv",
    "exp3e_lstm_grid_matched_vs_raw_dense_summary": FH_PLOTS / "experiment3e_lstm_grid_matched_vs_raw_dense_summary.csv",
    "exp3_lstm_combined_key_contrast_summary": FH_PLOTS / "experiment3_lstm_combined_key_contrast_summary.csv",
    "exp3_lstm_final_contrast_coverage": FH_PLOTS / "experiment3_lstm_final_contrast_coverage.csv",

    "exp3_grud_run_results": FH_PLOTS / "experiment3_grud_condition_run_results.csv",
    "exp3_grud_across_seed_summary": FH_PLOTS / "experiment3_grud_across_seed_summary.csv",
    "exp3_grud_structured_shift_summary": FH_PLOTS / "experiment3_grud_structured_shift_summary.csv",
    "exp3_grud_normalizer_control_summary": FH_PLOTS / "experiment3_grud_normalizer_control_summary.csv",
    "exp3_grud_random_shift_summary": FH_PLOTS / "experiment3_grud_random_shift_summary.csv",
    "exp3_grud_horizon_replication_summary": FH_PLOTS / "experiment3_grud_horizon_replication_summary.csv",
    "exp3_grud_combined_key_contrast_summary": FH_PLOTS / "experiment3_grud_combined_key_contrast_summary.csv",
    "exp3_grud_cross_experiment_summary": FH_PLOTS / "experiment3_grud_cross_experiment_mismatch_summary.csv",
    "exp3_grud_final_status": FH_PLOTS / "experiment3_grud_final_status.csv",
    "exp3_grud_final_contrast_coverage": FH_PLOTS / "experiment3_grud_final_contrast_coverage.csv",
}

DATA = {}
load_rows = []
for key, path in EXPORTS.items():
    if path.exists():
        df = pd.read_csv(path)
        DATA[key] = df
        load_rows.append({"key": key, "status": "LOADED", "rows": len(df), "path": str(path)})
    else:
        DATA[key] = pd.DataFrame()
        load_rows.append({"key": key, "status": "NOT AVAILABLE", "rows": 0, "path": str(path)})

load_status = pd.DataFrame(load_rows)
display(load_status)


In [ ]:
def parse_listish(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    text = str(value)
    try:
        parsed = ast.literal_eval(text)
        return parsed if isinstance(parsed, list) else [parsed]
    except Exception:
        return [x.strip() for x in text.strip("[]").split(",") if x.strip()]


def status_from_n(n, expected=5):
    if pd.isna(n) or int(n) == 0:
        return "NOT AVAILABLE"
    return "COMPLETE" if int(n) >= int(expected) else "PARTIAL"


def add_status(df, n_col="n", expected=5):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    if n_col in out.columns:
        out["expected_seeds"] = out.get("expected_seeds", expected)
        out["status"] = out[n_col].apply(lambda x: status_from_n(x, expected))
    elif "n_seeds" in out.columns:
        out["expected_seeds"] = out.get("expected_seeds", expected)
        out["status"] = out["n_seeds"].apply(lambda x: status_from_n(x, expected))
    elif "complete_seed_set" in out.columns:
        out["status"] = out["complete_seed_set"].map(lambda x: "COMPLETE" if bool(x) else "PARTIAL")
    return out


def select_cols(df, cols):
    return df[[c for c in cols if c in df.columns]] if len(df) else df


def show_table(title, df, cols=None, max_rows=120):
    print("\n" + title)
    print("=" * len(title))
    if df is None or len(df) == 0:
        print("NOT AVAILABLE")
        return
    view = select_cols(df, cols) if cols else df
    display(view.head(max_rows))
    if len(view) > max_rows:
        print("... {} more rows".format(len(view) - max_rows))


def yerr_from_ci(df, mean_col="mean", low_col="ci95_low", high_col="ci95_high"):
    if not {mean_col, low_col, high_col}.issubset(df.columns):
        return None
    low = df[mean_col].astype(float) - df[low_col].astype(float)
    high = df[high_col].astype(float) - df[mean_col].astype(float)
    return np.vstack([low.fillna(0), high.fillna(0)])


def plot_ci(df, x, y="mean", hue=None, title="", ylabel="", xlabel="", low="ci95_low", high="ci95_high"):
    if df is None or len(df) == 0 or x not in df.columns or y not in df.columns:
        print("NOT AVAILABLE:", title)
        return
    fig, ax = plt.subplots(figsize=(8, 4.5))
    groups = [(None, df)] if hue is None or hue not in df.columns else list(df.groupby(hue))
    for label, g in groups:
        g = g.sort_values(x)
        xs = g[x].values
        ys = g[y].astype(float).values
        yerr = yerr_from_ci(g, y, low, high)
        ax.errorbar(xs, ys, yerr=yerr, marker="o", capsize=4, linewidth=2, label=None if label is None else str(label))
        n_col = "n" if "n" in g.columns else "n_seeds" if "n_seeds" in g.columns else None
        if n_col:
            for xi, yi, ni in zip(xs, ys, g[n_col].values):
                ax.annotate("n={}".format(ni), (xi, yi), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
    ax.axhline(0, color="black", linewidth=0.8, alpha=0.4) if "degradation" in title.lower() or "shift" in title.lower() or "contrast" in title.lower() else None
    ax.set_title(title)
    ax.set_xlabel(xlabel or x)
    ax.set_ylabel(ylabel or y)
    if hue and hue in df.columns:
        ax.legend(title=hue, bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()


def plot_bar_effects(df, x="metric", y="mean_difference", hue=None, title="", ylabel="effect"):
    if df is None or len(df) == 0 or x not in df.columns:
        print("NOT AVAILABLE:", title)
        return
    y = y if y in df.columns else "effect" if "effect" in df.columns else "mean" if "mean" in df.columns else None
    if y is None:
        print("NOT AVAILABLE:", title)
        return
    fig, ax = plt.subplots(figsize=(9, 4.5))
    work = df.copy()
    work[x] = work[x].astype(str)
    if hue and hue in work.columns:
        labels = sorted(work[x].unique())
        hues = sorted(work[hue].astype(str).unique())
        width = 0.8 / max(1, len(hues))
        base = np.arange(len(labels))
        for i, h in enumerate(hues):
            sub = work[work[hue].astype(str) == h]
            vals = [sub[sub[x] == lab][y].astype(float).mean() if len(sub[sub[x] == lab]) else np.nan for lab in labels]
            ax.bar(base + (i - (len(hues)-1)/2)*width, vals, width, label=h)
        ax.set_xticks(base)
        ax.set_xticklabels(labels)
        ax.legend(title=hue, bbox_to_anchor=(1.02, 1), loc="upper left")
    else:
        ax.bar(work[x].values, work[y].astype(float).values)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()


## 0. Experiment Coverage Dashboard

**What is shown:** one compact status table built from the exported source dataframes.

**How each value is computed:** rows come from run-level files when available; observed seeds are grouped by experiment condition or contrast. If a source only exports an already summarized table, its `n` or `n_seeds` is used directly.

**How to read columns:** `n` is observed model seeds; `expected_seeds` is the planned five-seed set unless a source says otherwise; `missing_seeds` is shown when it can be inferred.

**Positive/negative values:** this dashboard does not report effects, only availability.

**Relevant pattern:** `COMPLETE` means all expected seeds are present, `PARTIAL` means some seeds exist, and `NOT AVAILABLE` means no export is currently present.

**Uncertainty interval:** none in this table.


In [ ]:
def coverage_from_run_df(df, experiment, group_cols, seed_col, extra=None):
    if df is None or len(df) == 0 or seed_col not in df.columns:
        row = {"experiment": experiment, "status": "NOT AVAILABLE", "n": 0, "expected_seeds": 5, "missing_seeds": sorted(EXPECTED_SEEDS)}
        if extra:
            row.update(extra)
        return [row]
    rows = []
    for keys, g in df.groupby(group_cols, dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        observed = set(pd.to_numeric(g[seed_col], errors="coerce").dropna().astype(int))
        row = dict(zip(group_cols, keys))
        row.update({
            "experiment": experiment,
            "observed_seeds": sorted(observed),
            "expected_seeds": 5,
            "n": len(observed),
            "missing_seeds": sorted(EXPECTED_SEEDS - observed),
            "status": status_from_n(len(observed)),
        })
        if extra:
            row.update(extra)
        rows.append(row)
    return rows

coverage_rows = []
coverage_rows += coverage_from_run_df(DATA["exp1_runs"], "Exp1", ["timestep"], "seed", {"task": "in-hospital mortality", "model": "LSTM", "comparison": "absolute performance"})
coverage_rows += coverage_from_run_df(DATA["exp2a_test_runs"], "Exp2A", ["horizon", "timestep"], "seed", {"task": "fixed-horizon ICU outcome", "model": "LSTM", "comparison": "1h vs coarser resolution"})
coverage_rows += coverage_from_run_df(DATA["exp2b_run_results"], "Exp2B/C", ["horizon", "r", "condition"], "model_seed", {"task": "fixed-horizon ICU outcome", "model": "LSTM", "comparison": "A/B/C/D mechanism"})
coverage_rows += coverage_from_run_df(DATA["exp3_lstm_run_results"], "Exp3 LSTM", ["experiment", "horizon", "target_r", "condition", "normalizer_regime"], "model_seed", {"task": "fixed-horizon ICU outcome", "model": "LSTM", "comparison": "frequency-shift absolute condition"})
coverage_rows += coverage_from_run_df(DATA["exp3_grud_run_results"], "Exp3 GRU-D", ["experiment", "horizon", "target_r", "condition", "normalizer_regime"], "model_seed", {"task": "fixed-horizon ICU outcome", "model": "GRU-D", "comparison": "frequency-shift absolute condition"})

coverage_dashboard = pd.DataFrame(coverage_rows)
coverage_cols = ["experiment", "task", "model", "horizon", "timestep", "target_r", "r", "condition", "normalizer_regime", "comparison", "observed_seeds", "expected_seeds", "n", "missing_seeds", "status"]
show_table("Experiment coverage dashboard", coverage_dashboard, coverage_cols, max_rows=300)


## 1. Experiment 1 - Does Measurement Frequency Matter?

### Scientific Question
Does prediction performance for in-hospital mortality deteriorate when EHR measurements are represented at coarser temporal resolutions?

### Experimental Design
Model: LSTM. Task: in-hospital mortality. Prediction time/horizon: the standard in-hospital mortality setup from the source notebook. Temporal resolution varies over the available timesteps, currently intended as 1h, 2h, 4h, 8h, 12h, and 24h. Training and test observation regimes match the same temporal resolution. Checkpoint selection is based on validation AUPRC; validation AUROC and test metrics are read at the same selected epoch.

### Quantity Being Measured
Absolute validation/test performance is `M(r,s)` for resolution `r` and model seed `s`. Degradation is `Delta(r,s) = M(1h,s) - M(r,s)`, computed within seed before summarizing across seeds.

Positive degradation means the coarser resolution performed worse than 1h. Zero means little or no difference. Negative means the coarser resolution performed better.

### Why This Comparison Is Informative
It tests whether reducing measurement frequency alone is associated with lower predictive performance, while preserving the source notebook's checkpoint and test-prediction provenance rules.

### Expected Evidence
If fine temporal information matters, degradation should become positive at sufficiently coarse resolutions. If frequency has little effect, performance and degradation should remain near flat. Non-monotonic behavior is possible and should be displayed.

### Caveats
Strict monotonic degradation is not required. Across-seed CIs are not paired patient-level tests. Partial seed sets have unstable across-seed uncertainty.


**Table/plot guide:** The next tables show validation and test AUPRC/AUROC means with 95% Student-t CIs across model seeds. Columns `mean`, `sd`, `sem`, and `ci95_*` are computed by the source notebook using model seeds as independent runs. `n` shows available seeds, so partial exports remain visible.


In [ ]:
for title, key in [
    ("Exp1 validation AUPRC", "exp1_val_auprc_summary"),
    ("Exp1 validation AUROC", "exp1_val_auroc_summary"),
    ("Exp1 test AUPRC", "exp1_test_auprc_summary"),
    ("Exp1 test AUROC", "exp1_test_auroc_summary"),
]:
    show_table(title, add_status(DATA[key]), ["timestep", "mean", "sd", "sem", "ci95_low", "ci95_high", "n", "status"])

plot_ci(add_status(DATA["exp1_test_auprc_summary"]), "timestep", title="Exp1 test AUPRC vs temporal resolution", ylabel="test AUPRC", xlabel="temporal resolution (hours)")
plot_ci(add_status(DATA["exp1_test_auroc_summary"]), "timestep", title="Exp1 test AUROC vs temporal resolution", ylabel="test AUROC", xlabel="temporal resolution (hours)")


**Degradation guide:** The degradation tables and plots show `1h - coarse`, computed within model seed and summarized with a 95% Student-t CI across seeds. Positive values mean degradation under coarser resolution. The interval is across model seeds, not a patient bootstrap interval. Partial status reflects how many paired seed-level drops are available.


In [ ]:
exp1_auprc_drop = add_status(DATA["exp1_auprc_drop_summary"])
exp1_auroc_drop = add_status(DATA["exp1_auroc_drop_summary"])
show_table("Exp1 AUPRC degradation from 1h", exp1_auprc_drop, ["timestep", "mean", "sd", "sem", "ci95_low", "ci95_high", "n", "status"])
show_table("Exp1 AUROC degradation from 1h", exp1_auroc_drop, ["timestep", "mean", "sd", "sem", "ci95_low", "ci95_high", "n", "status"])
plot_ci(exp1_auprc_drop, "timestep", title="Exp1 AUPRC degradation from 1h", ylabel="1h - coarse AUPRC", xlabel="temporal resolution (hours)")
plot_ci(exp1_auroc_drop, "timestep", title="Exp1 AUROC degradation from 1h", ylabel="1h - coarse AUROC", xlabel="temporal resolution (hours)")


## 2. Experiment 2A - Does Frequency Matter More For Short Horizons?

### Scientific Question
At a fixed prediction time of 24h after ICU admission, is the cost of coarser measurement frequency larger for short prediction horizons than for long horizons?

### Experimental Design
Model: LSTM. Task: fixed-horizon ICU outcome from the source notebook. Prediction time: 24h after ICU admission. Prediction horizons: 12h, 24h, 48h, 96h, and 168h when available. Temporal resolution varies across available source timesteps. Training and test observation regimes match the resolution being evaluated. Checkpoints are selected by validation AUPRC.

### Quantity Being Measured
Absolute performance is `M(h,r,s)`. Degradation is `Delta(h,r,s) = M(h,1h,s) - M(h,r,s)`, computed within seed.

Positive degradation means the coarser resolution performed worse than 1h for that horizon. For Brier bootstrap contrasts, the source reverses the raw lower-is-better metric so positive still means the 1h/reference condition is better.

### Why This Comparison Is Informative
It asks whether temporal detail is especially important for near-term prediction rather than equally important across all horizons.

### Expected Evidence
If short-horizon prediction depends more on fine temporal information, degradation should be larger at 12h than at longer horizons. If the effect is horizon-independent, degradation should be similar across horizons. Non-monotonic patterns should be displayed.

### Caveats
Raw AUPRC cannot be compared directly across horizons because outcome prevalence changes. Across-seed CIs and paired patient-bootstrap CIs are different uncertainty estimates.


**Validation/test performance guide:** These tables and plots show absolute AUPRC/AUROC by horizon and temporal resolution. Values are source-computed means across model seeds with Student-t CIs. `n` is the number of seeds, so partial curves may appear. Raw AUPRC across horizons reflects different prevalence and should be interpreted cautiously.


In [ ]:
for title, key in [
    ("Exp2A validation AUPRC", "exp2a_val_auprc_summary"),
    ("Exp2A validation AUROC", "exp2a_val_auroc_summary"),
    ("Exp2A test AUPRC", "exp2a_test_auprc_summary"),
    ("Exp2A test AUROC", "exp2a_test_auroc_summary"),
]:
    show_table(title, add_status(DATA[key]), ["horizon", "timestep", "mean", "sd", "sem", "ci95_low", "ci95_high", "n", "status"])

plot_ci(add_status(DATA["exp2a_test_auprc_summary"]), "timestep", hue="horizon", title="Exp2A test AUPRC vs resolution by horizon", ylabel="test AUPRC", xlabel="temporal resolution (hours)")
plot_ci(add_status(DATA["exp2a_test_auroc_summary"]), "timestep", hue="horizon", title="Exp2A test AUROC vs resolution by horizon", ylabel="test AUROC", xlabel="temporal resolution (hours)")


**Degradation and bootstrap guide:** Degradation summaries show `1h - coarse` with Student-t CIs across seeds. The patient-bootstrap tables come from paired test predictions and include patient-level bootstrap CIs. Positive values mean the 1h/reference condition performed better; for Brier, positive means lower Brier for the reference.


In [ ]:
for title, key in [
    ("Exp2A validation AUPRC degradation", "exp2a_val_auprc_drop_summary"),
    ("Exp2A validation AUROC degradation", "exp2a_val_auroc_drop_summary"),
    ("Exp2A test AUPRC degradation", "exp2a_test_auprc_drop_summary"),
    ("Exp2A test AUROC degradation", "exp2a_test_auroc_drop_summary"),
]:
    show_table(title, add_status(DATA[key]), ["horizon", "timestep", "mean", "sd", "sem", "ci95_low", "ci95_high", "n", "status"])

plot_ci(add_status(DATA["exp2a_test_auprc_drop_summary"]), "horizon", hue="timestep", title="Exp2A test AUPRC degradation vs horizon", ylabel="1h - coarse AUPRC", xlabel="prediction horizon (hours)")
plot_ci(add_status(DATA["exp2a_test_auroc_drop_summary"]), "horizon", hue="timestep", title="Exp2A test AUROC degradation vs horizon", ylabel="1h - coarse AUROC", xlabel="prediction horizon (hours)")

exp2a_boot = add_status(DATA["exp2a_bootstrap_summary"], n_col="n_seeds")
show_table("Exp2A across-seed paired patient-bootstrap summary", exp2a_boot, ["horizon", "timestep", "metric", "mean_difference", "sd_difference", "seed_ci95_low", "seed_ci95_high", "patient_bootstrap_ci_low", "patient_bootstrap_ci_high", "n_seeds", "missing_seeds", "status"], max_rows=300)


## 3-4. Experiment 2B/C - Why Does Frequency Matter, And Does The Mechanism Depend On Severity?

### Scientific Question
Which parts of the frequency effect are associated with observation loss, temporal placement, and representation/grid changes? Does that mechanism change as thinning becomes more severe?

### Experimental Design
Model: LSTM. Task: fixed-horizon ICU outcome. Prediction time and horizons follow the source notebook. Conditions are preserved exactly from the source:

- A: native observations represented on the standard 1h grid.
- B: structured r-hour measurement selection/thinning; retained observations keep original timestamps and are represented on the downstream 1h grid.
- C: random matched-count thinning; for each patient-variable, retain the same effective number of observations as B but choose positions randomly from originally occupied 1h cells. Downstream representation remains 1h.
- D: standard r-hour discretization using the ordinary r-hour grid and therefore fewer recurrent steps.

### Quantity Being Measured
Absolute performance compares A/B/C/D condition metrics. Primary contrasts are A-vs-B, B-vs-C, and B-vs-D. Positive contrast values mean the left condition is better; for Brier the source reverses sign so positive still means the left/reference condition has lower Brier.

### Why This Comparison Is Informative
A-vs-B isolates structured observation removal relative to native observations. B-vs-C asks whether temporal pattern matters after matching observation count. B-vs-D asks whether the coarse representation/grid changes performance beyond sparse observation selection.

### Expected Evidence
Increasing A-B with `r` suggests stronger information loss at lower observation frequency. Increasing B-C suggests temporal structure matters more with stronger thinning. Increasing B-D suggests representation/grid choice matters more. Flat curves suggest relative insensitivity to severity.

### Caveats
A/B/C/D are not an additive causal decomposition. B-vs-D is not a pure sequence-length contrast. Use only available `r` values; do not fabricate missing severities.


**Absolute condition guide:** The next table contains all exported A/B/C/D condition-level performance. Means and CIs are across model seeds. Brier raw values are lower-is-better here; only contrast tables use the positive-is-left-better sign convention. Completeness is based on available seed counts.


In [ ]:
exp2b_cond = add_status(DATA["exp2b_condition_summary"])
show_table("Exp2B/C A/B/C/D condition-level summary", exp2b_cond, ["horizon", "r", "condition", "metric", "mean", "sd", "sem", "ci95_low", "ci95_high", "n", "status"], max_rows=300)
for metric in ["test_auprc", "test_auroc", "test_brier"]:
    sub = exp2b_cond[exp2b_cond.get("metric", pd.Series(dtype=str)).astype(str).eq(metric)] if len(exp2b_cond) else pd.DataFrame()
    plot_ci(sub, "condition", hue="r", title="Exp2B/C {} by A/B/C/D condition".format(metric), ylabel=metric, xlabel="condition")


**Contrast guide:** The contrast tables show seed-level paired patient-bootstrap contrasts and across-seed summaries. Positive means the left condition is better; for Brier, positive means lower Brier for the left/reference condition. Patient-bootstrap CIs and Student-t seed CIs must be read separately.


In [ ]:
exp2b_contrasts = add_status(DATA["exp2b_across_seed_summary"], n_col="n_seeds")
show_table("Exp2B primary mechanism contrasts", exp2b_contrasts, ["horizon", "r", "contrast", "left_condition", "right_condition", "primary", "metric", "mean_difference", "sd_difference", "seed_ci95_low", "seed_ci95_high", "patient_bootstrap_ci_low", "patient_bootstrap_ci_high", "n_seeds", "missing_seeds", "status"], max_rows=300)
for metric in METRICS:
    sub = exp2b_contrasts[exp2b_contrasts.get("metric", pd.Series(dtype=str)).eq(metric)] if len(exp2b_contrasts) else pd.DataFrame()
    plot_ci(sub.rename(columns={"mean_difference": "mean", "seed_ci95_low": "ci95_low", "seed_ci95_high": "ci95_high", "n_seeds": "n"}), "r", hue="contrast", title="Exp2C {} mechanism effect vs r".format(metric.upper()), ylabel="left condition better (+)", xlabel="r")


## 5-11. Experiment 3 LSTM - Observation-Frequency Distribution Shift

### Scientific Question
How much held-out performance is recovered when the training observation regime matches the deployment observation regime, and do controls support that interpretation?

### Experimental Design
Model: LSTM. Task: fixed-horizon ICU outcome. Exp3A uses structured observation-frequency shift at h=12 for r=4 and r=8. Exp3B is a normalizer control at h=12, r=4. Exp3C uses random matched-count thinning at h=12, r=4, sampling seed 100. Exp3D repeats structured r=4 at h=96. Exp3E repeats raw-representation analyses for r=4 and r=8 and compares grid versus raw shift sensitivity.

### Quantity Being Measured
Absolute performance is condition-level AUPRC/AUROC/Brier by train regime, test regime, horizon, r, and normalizer regime. Main shift recovery for AUPRC/AUROC is `M(matched sparse -> sparse) - M(dense -> sparse)`. For Brier, the source uses `Brier(dense -> sparse) - Brier(matched sparse -> sparse)`, so positive consistently means improvement from matching the training observation regime.

### Why This Comparison Is Informative
It separates information loss at deployment from train/test observation-regime mismatch. The normalizer control asks whether normalization statistics explain the effect. Random matched thinning asks whether the mismatch persists without periodic structure. The h=96 replication links to Exp2A. Raw/grid analyses ask whether the fixed grid amplifies sensitivity.

### Expected Evidence
Positive mismatch recovery suggests matched sparse training improves sparse deployment performance. Little normalizer-control difference suggests normalizer choice alone does not explain recovery. Structured-vs-random and r4-vs-r8 comparisons describe whether temporal pattern and severity matter. Grid-minus-raw above zero suggests the fixed grid amplifies the shift penalty.

### Caveats
A difference between regimes does not prove universal model robustness. LSTM Exp3 exports may be partial or unavailable if the source notebook run was interrupted. Do not call partial results final.


**Absolute performance guide:** These are raw condition-level test metrics summarized across seeds with Student-t CIs. Brier is lower-is-better in absolute tables. `status` flags partial seed availability. No patient-bootstrap interval is shown in this absolute table.


In [ ]:
exp3_abs = add_status(DATA["exp3_lstm_absolute_summary"])
show_table("Experiment 3 LSTM absolute performance", exp3_abs, ["experiment", "horizon", "target_r", "condition", "normalizer_regime", "metric", "mean", "sd", "sem", "ci95_low", "ci95_high", "n", "status"], max_rows=300)
plot_ci(exp3_abs[exp3_abs.get("metric", pd.Series(dtype=str)).astype(str).str.contains("auprc|test_auprc", regex=True)] if len(exp3_abs) else pd.DataFrame(), "condition", hue="target_r", title="Exp3 LSTM absolute AUPRC by condition", ylabel="AUPRC", xlabel="condition")


**Exp3A-D contrast guide:** The following tables come directly from the source LSTM summaries. Positive values mean the left/matched condition performs better. `seed_ci95_*` is a Student-t CI across model seeds. `patient_bootstrap_ci_*` is a paired patient-level bootstrap CI. Both are shown when exported.


In [ ]:
for title, key in [
    ("Exp3A LSTM structured shift recovery", "exp3_lstm_structured_shift_summary"),
    ("Exp3B LSTM normalizer control", "exp3_lstm_normalizer_control_summary"),
    ("Exp3C LSTM random matched-count shift", "exp3_lstm_random_shift_summary"),
    ("Exp3D LSTM longer-horizon replication", "exp3_lstm_horizon_replication_summary"),
    ("Exp3 combined key-contrast summary", "exp3_lstm_combined_key_contrast_summary"),
    ("Exp3 final contrast coverage", "exp3_lstm_final_contrast_coverage"),
]:
    df = add_status(DATA[key], n_col="n_seeds")
    show_table(title, df, ["experiment", "contrast", "horizon", "r", "metric", "left_condition", "right_condition", "mean_difference", "sd_difference", "seed_ci95_low", "seed_ci95_high", "patient_bootstrap_ci_low", "patient_bootstrap_ci_high", "n_seeds", "expected_seeds", "missing_seeds", "status", "present_in_final_summary", "final_reporting_ready"], max_rows=300)

lstm_combined = add_status(DATA["exp3_lstm_combined_key_contrast_summary"], n_col="n_seeds")
plot_ci(lstm_combined.rename(columns={"mean_difference": "mean", "seed_ci95_low": "ci95_low", "seed_ci95_high": "ci95_high", "n_seeds": "n"}), "r", hue="contrast", title="Exp3 LSTM mismatch/control effects by r", ylabel="left/matched better (+)", xlabel="r")


**Exp3E raw/grid guide:** Raw mismatch recovery compares raw matched sparse training against raw dense-trained sparse deployment. Grid-vs-raw reports `Delta_grid - Delta_raw`; positive means the fixed-grid representation has a larger frequency-shift penalty than raw. The diagnostic compares grid matched-frequency against raw dense-trained as a descriptive robustness check, not an isolated causal contrast.


In [ ]:
for title, key in [
    ("Exp3E LSTM raw condition-level performance", "exp3e_lstm_raw_absolute_summary"),
    ("Exp3E LSTM raw mismatch recovery", "exp3e_lstm_raw_shift_summary"),
    ("Exp3E LSTM grid-vs-raw difference-of-shifts", "exp3_lstm_grid_vs_raw_shift_summary"),
    ("Exp3E LSTM grid-matched vs raw-dense diagnostic", "exp3e_lstm_grid_matched_vs_raw_dense_summary"),
]:
    df = add_status(DATA[key], n_col="n_seeds")
    show_table(title, df, max_rows=300)

plot_ci(add_status(DATA["exp3e_lstm_raw_shift_summary"], n_col="n_seeds").rename(columns={"mean_difference": "mean", "seed_ci95_low": "ci95_low", "seed_ci95_high": "ci95_high", "n_seeds": "n"}), "r", hue="metric", title="Exp3E raw mismatch recovery", ylabel="matched raw better (+)", xlabel="r")
plot_bar_effects(add_status(DATA["exp3_lstm_grid_vs_raw_shift_summary"], n_col="n_seeds"), x="r", y="mean_grid_minus_raw_shift", hue="metric", title="Exp3E grid minus raw shift sensitivity", ylabel="Delta_grid - Delta_raw")


## 12. GRU-D Experiment 3

### Scientific Question
Does explicitly modeling observation masks and elapsed time with GRU-D change sensitivity to observation-frequency shift?

### Experimental Design
Model: GRU-D. Task and Exp3A-D definitions follow the LSTM Exp3 notebook. Implemented analyses include structured shift at h=12 r=4/r=8, normalizer control at h=12 r=4, random matched-count h=12 r=4 sampling seed 100, and h=96 r=4 structured replication.

### Quantity Being Measured
Absolute performance and mismatch/control contrasts use the same orientation as LSTM Exp3. For AUPRC/AUROC, positive recovery means matched sparse training performs better on sparse deployment. For Brier, positive means matched training has lower Brier because the source reports mismatched minus matched.

### Why This Comparison Is Informative
GRU-D uses masks and elapsed time, so it may respond differently to observation-frequency shifts than an LSTM on gridded inputs.

### Expected Evidence
Smaller, larger, or similar GRU-D effects relative to LSTM are all informative. Do not conclude universal superiority from one point estimate, especially under partial seeds.

### Caveats
GRU-D runs may still be in progress. Partial result sets are valid exploratory outputs but not final evidence.


**GRU-D tables/plots guide:** Absolute tables use Student-t CIs across model seeds. Contrast summaries show both seed CIs and patient-bootstrap CIs when exported. Partial `n` is displayed instead of suppressing incomplete analyses.


In [ ]:
for title, key in [
    ("GRU-D absolute condition performance", "exp3_grud_across_seed_summary"),
    ("GRU-D Exp3A structured shift", "exp3_grud_structured_shift_summary"),
    ("GRU-D Exp3B normalizer control", "exp3_grud_normalizer_control_summary"),
    ("GRU-D Exp3C random matched-count shift", "exp3_grud_random_shift_summary"),
    ("GRU-D Exp3D horizon replication", "exp3_grud_horizon_replication_summary"),
    ("GRU-D combined key-contrast summary", "exp3_grud_combined_key_contrast_summary"),
    ("GRU-D final status", "exp3_grud_final_status"),
    ("GRU-D final contrast coverage", "exp3_grud_final_contrast_coverage"),
]:
    df = add_status(DATA[key], n_col="n_seeds")
    show_table(title, df, max_rows=300)

grud_combined = add_status(DATA["exp3_grud_combined_key_contrast_summary"], n_col="n_seeds")
plot_ci(grud_combined.rename(columns={"mean_difference": "mean", "seed_ci95_low": "ci95_low", "seed_ci95_high": "ci95_high", "n_seeds": "n"}), "r", hue="contrast", title="GRU-D Exp3 mismatch/control effects by r", ylabel="left/matched better (+)", xlabel="r")


## 13-17. Cross-Experiment Comparisons

### Scientific Questions
These exploratory comparisons connect the individual experiments:

- LSTM vs GRU-D: does mask/time modeling alter frequency-shift sensitivity?
- Structured vs random matched thinning: how much effect is due to fewer observations versus temporal removal pattern?
- r4 vs r8 severity: does more severe frequency reduction create larger information loss or mismatch recovery?
- Short vs long horizon: does h=12 differ from h=96, linking back to Exp2A?
- Grid vs raw representation: does hourly gridding amplify the shift penalty?

### Quantity Being Measured
All effect sizes use source summaries. For Exp3 contrasts, positive means matched/left/reference condition is better; Brier is oriented so positive means improvement. For grid-vs-raw, positive `grid - raw` means a larger shift penalty in the grid representation.

### Caveats
These are side-by-side exploratory comparisons. They do not require complete five-seed results for both models and should not be overinterpreted as architecture rankings.


In [ ]:
def model_effect_table(lstm, grud):
    rows = []
    common_specs = [
        ("structured r4 h12", "Exp3A Structured frequency shift", 12, 4, "structured shift"),
        ("structured r8 h12", "Exp3A Structured frequency shift", 12, 8, "structured shift"),
        ("normalizer control", "Exp3B Normalizer control", 12, 4, "normalizer control"),
        ("random matched r4 h12", "Exp3C Random-matched frequency shift", 12, 4, "random-matched shift"),
        ("structured r4 h96", "Exp3D Horizon replication", 96, 4, "structured shift"),
    ]
    for label, experiment, horizon, r, contrast in common_specs:
        for metric in METRICS:
            row = {"comparison": label, "experiment": experiment, "horizon": horizon, "r": r, "metric": metric}
            for model, df in [("LSTM", lstm), ("GRU-D", grud)]:
                sub = df[
                    (df.get("experiment", pd.Series(dtype=str)).astype(str) == experiment) &
                    (pd.to_numeric(df.get("horizon", pd.Series(dtype=float)), errors="coerce") == horizon) &
                    (pd.to_numeric(df.get("r", pd.Series(dtype=float)), errors="coerce") == r) &
                    (df.get("contrast", pd.Series(dtype=str)).astype(str) == contrast) &
                    (df.get("metric", pd.Series(dtype=str)).astype(str) == metric)
                ] if len(df) else pd.DataFrame()
                prefix = model.lower().replace("-", "")
                if len(sub) == 1:
                    one = sub.iloc[0]
                    row.update({
                        f"{prefix}_effect": one.get("mean_difference", np.nan),
                        f"{prefix}_seed_ci_low": one.get("seed_ci95_low", np.nan),
                        f"{prefix}_seed_ci_high": one.get("seed_ci95_high", np.nan),
                        f"{prefix}_patient_ci_low": one.get("patient_bootstrap_ci_low", np.nan),
                        f"{prefix}_patient_ci_high": one.get("patient_bootstrap_ci_high", np.nan),
                        f"{prefix}_n": one.get("n_seeds", np.nan),
                        f"{prefix}_status": status_from_n(one.get("n_seeds", 0)),
                    })
                else:
                    row.update({f"{prefix}_status": "NOT AVAILABLE"})
            rows.append(row)
    return pd.DataFrame(rows)

lstm_combined = DATA["exp3_lstm_combined_key_contrast_summary"].copy()
grud_combined = DATA["exp3_grud_combined_key_contrast_summary"].copy()
lstm_vs_grud = model_effect_table(lstm_combined, grud_combined)
show_table("LSTM vs GRU-D common frequency-shift effects", lstm_vs_grud, max_rows=200)

structured_random = lstm_vs_grud[lstm_vs_grud["comparison"].isin(["structured r4 h12", "random matched r4 h12"])]
show_table("Structured vs random matched thinning", structured_random, max_rows=100)
plot_bar_effects(structured_random.rename(columns={"lstm_effect": "effect"}), x="comparison", y="effect", hue="metric", title="LSTM structured vs random matched r4", ylabel="effect")
plot_bar_effects(structured_random.rename(columns={"grud_effect": "effect"}), x="comparison", y="effect", hue="metric", title="GRU-D structured vs random matched r4", ylabel="effect")

severity = lstm_vs_grud[lstm_vs_grud["comparison"].isin(["structured r4 h12", "structured r8 h12"])]
show_table("r4 vs r8 structured severity comparison", severity, max_rows=100)

horizon_cmp = lstm_vs_grud[lstm_vs_grud["comparison"].isin(["structured r4 h12", "structured r4 h96"])]
show_table("Short vs long horizon structured r4 comparison", horizon_cmp, max_rows=100)

show_table("Grid vs raw representation comparison", add_status(DATA["exp3_lstm_grid_vs_raw_shift_summary"], n_col="n_seeds"), ["r", "metric", "mean_grid_shift", "mean_raw_shift", "mean_grid_minus_raw_shift", "n_seeds", "status"], max_rows=100)


## 18. Master Everything-Tested Table

**What is shown:** every important exported scientific contrast from Exp1, Exp2A, Exp2B/C, Exp3 LSTM, Exp3E, and GRU-D Exp3.

**How each value is computed:** rows are not recomputed here. They are reshaped from source notebook CSV exports. Effect orientation is recorded explicitly.

**How to read columns:** `effect` is the source point estimate; `seed_ci_*` is the Student-t CI across model seeds; `patient_ci_*` is the paired patient-bootstrap CI when available; `n_seeds`, `expected_n_seeds`, `missing_seeds`, and `status` describe completeness.

**Positive/negative values:** Exp1/Exp2A positive means degradation under coarser resolution. Exp2B/C positive means the left condition is better. Exp3 positive means matched/left condition improves performance; Brier is oriented positive-is-improvement. Grid-vs-raw positive means grid shift exceeds raw shift.

**Relevant pattern:** this table is meant for scanning what has actually been tested and which results are partial.

**Uncertainty interval:** both uncertainty types are preserved in separate columns and may be missing when the source did not compute them.


In [ ]:
master_rows = []

def add_master(row):
    base = {
        "experiment": None, "subexperiment": None, "task": None, "model": None,
        "prediction_time": None, "horizon": None, "resolution_or_r": None,
        "train_regime": None, "test_regime": None, "normalizer_regime": None,
        "comparison": None, "metric": None, "effect": np.nan,
        "effect_orientation": None,
        "seed_ci_low": np.nan, "seed_ci_high": np.nan,
        "patient_ci_low": np.nan, "patient_ci_high": np.nan,
        "n_seeds": np.nan, "expected_n_seeds": 5, "missing_seeds": None,
        "status": "NOT AVAILABLE", "source_notebook": None,
    }
    base.update(row)
    if pd.notna(base.get("n_seeds", np.nan)):
        base["status"] = status_from_n(base["n_seeds"], base.get("expected_n_seeds", 5))
    master_rows.append(base)

for metric, df, source in [
    ("auprc", DATA["exp1_auprc_drop_summary"], "visualize_exp1_frequency.ipynb"),
    ("auroc", DATA["exp1_auroc_drop_summary"], "visualize_exp1_frequency.ipynb"),
]:
    for _, r in df.iterrows() if len(df) else []:
        add_master({"experiment": "Exp1", "subexperiment": "degradation from 1h", "task": "in-hospital mortality", "model": "LSTM", "resolution_or_r": r.get("timestep"), "comparison": "1h - coarse", "metric": metric, "effect": r.get("mean"), "effect_orientation": "positive = degradation under coarser resolution", "seed_ci_low": r.get("ci95_low"), "seed_ci_high": r.get("ci95_high"), "n_seeds": r.get("n"), "source_notebook": source})

for metric, df in [("auprc", DATA["exp2a_test_auprc_drop_summary"]), ("auroc", DATA["exp2a_test_auroc_drop_summary"])]:
    for _, r in df.iterrows() if len(df) else []:
        add_master({"experiment": "Exp2A", "subexperiment": "test degradation from 1h", "task": "fixed-horizon ICU outcome", "model": "LSTM", "prediction_time": "24h after ICU admission", "horizon": r.get("horizon"), "resolution_or_r": r.get("timestep"), "comparison": "1h - coarse", "metric": metric, "effect": r.get("mean"), "effect_orientation": "positive = degradation under coarser resolution", "seed_ci_low": r.get("ci95_low"), "seed_ci_high": r.get("ci95_high"), "n_seeds": r.get("n"), "source_notebook": "visualize_exp2a_horizon.ipynb"})

for _, r in DATA["exp2a_bootstrap_summary"].iterrows() if len(DATA["exp2a_bootstrap_summary"]) else []:
    add_master({"experiment": "Exp2A", "subexperiment": "paired patient bootstrap", "task": "fixed-horizon ICU outcome", "model": "LSTM", "prediction_time": "24h after ICU admission", "horizon": r.get("horizon"), "resolution_or_r": r.get("timestep"), "comparison": "1h - coarse", "metric": r.get("metric"), "effect": r.get("mean_difference"), "effect_orientation": "positive = 1h/reference better; Brier reversed", "seed_ci_low": r.get("seed_ci95_low"), "seed_ci_high": r.get("seed_ci95_high"), "patient_ci_low": r.get("patient_bootstrap_ci_low"), "patient_ci_high": r.get("patient_bootstrap_ci_high"), "n_seeds": r.get("n_seeds"), "missing_seeds": r.get("missing_seeds"), "source_notebook": "visualize_exp2a_horizon.ipynb"})

for _, r in DATA["exp2b_across_seed_summary"].iterrows() if len(DATA["exp2b_across_seed_summary"]) else []:
    add_master({"experiment": "Exp2B/C", "subexperiment": "A/B/C/D mechanism", "task": "fixed-horizon ICU outcome", "model": "LSTM", "horizon": r.get("horizon"), "resolution_or_r": r.get("r"), "train_regime": r.get("left_condition"), "test_regime": r.get("right_condition"), "comparison": r.get("contrast"), "metric": r.get("metric"), "effect": r.get("mean_difference"), "effect_orientation": "positive = left condition better; Brier reversed", "seed_ci_low": r.get("seed_ci95_low"), "seed_ci_high": r.get("seed_ci95_high"), "patient_ci_low": r.get("patient_bootstrap_ci_low"), "patient_ci_high": r.get("patient_bootstrap_ci_high"), "n_seeds": r.get("n_seeds"), "missing_seeds": r.get("missing_seeds"), "source_notebook": "visualize_exp2b_matched_count.ipynb"})

for key, model, source in [
    ("exp3_lstm_combined_key_contrast_summary", "LSTM", "visualize_exp3_frequency_shift.ipynb"),
    ("exp3_grud_combined_key_contrast_summary", "GRU-D", "visualize_exp3_grud_frequency_shift.ipynb"),
]:
    df = DATA[key]
    for _, r in df.iterrows() if len(df) else []:
        add_master({"experiment": "Exp3", "subexperiment": r.get("experiment"), "task": "fixed-horizon ICU outcome", "model": model, "horizon": r.get("horizon"), "resolution_or_r": r.get("r"), "train_regime": r.get("left_condition"), "test_regime": r.get("right_condition"), "normalizer_regime": "{} vs {}".format(r.get("left_normalizer_regime"), r.get("right_normalizer_regime")), "comparison": r.get("contrast"), "metric": r.get("metric"), "effect": r.get("mean_difference"), "effect_orientation": "positive = left/matched condition better; Brier reversed", "seed_ci_low": r.get("seed_ci95_low"), "seed_ci_high": r.get("seed_ci95_high"), "patient_ci_low": r.get("patient_bootstrap_ci_low"), "patient_ci_high": r.get("patient_bootstrap_ci_high"), "n_seeds": r.get("n_seeds"), "missing_seeds": r.get("missing_seeds"), "source_notebook": source})

for _, r in DATA["exp3_lstm_grid_vs_raw_shift_summary"].iterrows() if len(DATA["exp3_lstm_grid_vs_raw_shift_summary"]) else []:
    add_master({"experiment": "Exp3E", "subexperiment": "grid vs raw representation", "task": "fixed-horizon ICU outcome", "model": "LSTM", "horizon": 12, "resolution_or_r": r.get("r"), "comparison": "Delta_grid - Delta_raw", "metric": r.get("metric"), "effect": r.get("mean_grid_minus_raw_shift"), "effect_orientation": "positive = grid shift penalty larger than raw", "n_seeds": r.get("n_seeds"), "source_notebook": "visualize_exp3_frequency_shift.ipynb"})

master_contrasts = pd.DataFrame(master_rows)
show_table("Master everything-tested contrast table", master_contrasts, max_rows=500)


## 19. Notes For Future Runs

The summary updates when the source notebooks are rerun and their final export cells write CSVs. Keep all experiment discovery, checkpoint selection, normalizer resolution, test inference, prediction loading, provenance checks, and patient bootstrap implementation in the source notebooks.

If a section is `NOT AVAILABLE`, rerun the corresponding source notebook through its export cell. If a section is `PARTIAL`, more seeds can be added and the same source export cell rerun; this dashboard will then pick up the new rows without code changes.
